In [11]:
import pandas as pd
df = pd.read_csv("sim.csv")

In [12]:
from collections import defaultdict
# Keep only necessary columns to reduce memory
df = df[['nameOrig', 'nameDest', 'amount', 'isFraud']]

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (6362620, 4)
      nameOrig     nameDest    amount  isFraud
0  C1231006815  M1979787155   9839.64        0
1  C1666544295  M2044282225   1864.28        0
2  C1305486145   C553264065    181.00        1
3   C840083671    C38997010    181.00        1
4  C2048537720  M1230701703  11668.14        0


In [14]:
# Get all unique accounts
accounts = sorted(set(df['nameOrig']) | set(df['nameDest']))

# Map account IDs to indices (required for Union-Find)
account_to_idx = {acc: i for i, acc in enumerate(accounts)}

print("Total unique accounts:", len(accounts))

Total unique accounts: 9073900


In [15]:
graph = defaultdict(list)

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']
    amount = row["amount"] 
    graph[sender].append((receiver, amount))

print("Graph built with nodes:", len(graph))

Graph built with nodes: 6353307


In [24]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0] * size
        self.size = [1] * size

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])  # Path compression
        return self.parent[x]

    def union(self, x, y):
        rootX = self.find(x)
        rootY = self.find(y)

        if rootX == rootY:
            return

        # Union by rank
        if self.rank[rootX] < self.rank[rootY]:
            self.parent[rootX] = rootY
            self.size[rootY] += self.size[rootX]
        elif self.rank[rootX] > self.rank[rootY]:
            self.parent[rootY] = rootX
            self.size[rootX] += self.size[rootY]
        else:
            self.parent[rootY] = rootX
            self.rank[rootX] += 1
            self.size[rootX] += self.size[rootY]

In [25]:
uf = UnionFind(len(accounts))

for _, row in df.iterrows():
    sender = row['nameOrig']
    receiver = row['nameDest']

    idx1 = account_to_idx[sender]
    idx2 = account_to_idx[receiver]

    uf.union(idx1, idx2)

In [26]:
components = defaultdict(list)

for acc in accounts:
    idx = account_to_idx[acc]
    root = uf.find(idx)
    components[root].append(acc)

print("Total clusters detected:", len(components))

Total clusters detected: 2711280


In [50]:
# Sort clusters by size
cluster_size = {}
for root, members in components.items():
    size = len(members)
    for acc in members:
        cluster_size[acc] = size

top5 = sorted(cluster_size.items(), key=lambda x: x[1], reverse=True)[:15]
print("Top 5 cluster sizes:")
for acc, size in top5:
    print(f"Account {acc} belongs to a cluster of size {size}")

Top 5 cluster sizes:
Account C1005754692 belongs to a cluster of size 121
Account C1007091734 belongs to a cluster of size 121
Account C1019533814 belongs to a cluster of size 121
Account C1027077713 belongs to a cluster of size 121
Account C1036327038 belongs to a cluster of size 121
Account C105198810 belongs to a cluster of size 121
Account C1054591790 belongs to a cluster of size 121
Account C1085158170 belongs to a cluster of size 121
Account C1119147995 belongs to a cluster of size 121
Account C1149226671 belongs to a cluster of size 121
Account C1159710789 belongs to a cluster of size 121
Account C1165424674 belongs to a cluster of size 121
Account C1171146548 belongs to a cluster of size 121
Account C1223487299 belongs to a cluster of size 121
Account C12333931 belongs to a cluster of size 121


In [30]:
#degree and aggregate statistics
in_degree = defaultdict(int)
out_degree = defaultdict(int)
total_sent = defaultdict(float)
total_received = defaultdict(float)
transaction_count = defaultdict(int)

for _, row in df.iterrows():
    sender, receiver = row['nameOrig'], row['nameDest']
    amt = row['amount']
    
    out_degree[sender] += 1
    in_degree[receiver] += 1
    total_sent[sender] += amt
    total_received[receiver] += amt
    transaction_count[sender] += 1
    transaction_count[receiver] += 1

In [60]:
index = 0
stack = []
on_stack = set() 
indices = {}
lowlink = {} 
sccs = [] 
def tarjan_dfs(v):
    global index
    indices[v] = index
    lowlink[v] = index
    index += 1
    stack.append(v)
    on_stack.add(v) 
    for neighbor,_ in graph[v]:
        if neighbor not in indices:
            tarjan_dfs(neighbor)
            lowlink[v] = min(lowlink[v], lowlink[neighbor])
        elif neighbor in on_stack:
            lowlink[v] = min(lowlink[v], indices[neighbor])
     # If v is root of SCC
    if lowlink[v] == indices[v]:
        component = []
        while True:
            node = stack.pop()
            on_stack.remove(node)
            component.append(node)
            if node == v:
                break
        sccs.append(component)
 
# Run Tarjan on all nodes
for node in list(graph.keys()):
    if node not in indices:
        tarjan_dfs(node) 
print("Total SCCs:", len(sccs))

Total SCCs: 15435437


In [61]:
scc_size = {} 
for comp in sccs:
    size = len(comp)
    for acc in comp:
        scc_size[acc] = size
 
cycle_flag = {acc: 0 for acc in accounts} 
for comp in sccs:
    if len(comp) > 1:
        for acc in comp:
            cycle_flag[acc] = 1

In [62]:
features = pd.DataFrame({
    'account': accounts,
    'in_degree': [in_degree.get(acc, 0) for acc in accounts],
    'out_degree': [out_degree.get(acc, 0) for acc in accounts],
    'total_sent': [total_sent.get(acc, 0) for acc in accounts],
    'total_received': [total_received.get(acc, 0) for acc in accounts],
    'scc_size': [scc_size.get(acc, 1) for acc in accounts],
    'cycle_flag': [cycle_flag.get(acc, 0) for acc in accounts]
})
 
features['sent_received_ratio'] = (
    features['total_sent'] / (features['total_received'] + 1)
)
 

In [63]:
features.head()

,account,in_degree,out_degree,total_sent,total_received,scc_size,cycle_flag,sent_received_ratio
0,C1000000639,0,1,244486.46,0.0,1,0,244486.46
1,C1000001337,0,1,3170.28,0.0,1,0,3170.28
2,C1000001725,0,1,8424.74,0.0,1,0,8424.74
3,C1000002591,0,1,261877.19,0.0,1,0,261877.19
4,C1000003372,0,1,20528.65,0.0,1,0,20528.65


In [64]:
features.isna().sum()

account                0
in_degree              0
out_degree             0
total_sent             0
total_received         0
scc_size               0
cycle_flag             0
sent_received_ratio    0
dtype: int64

In [65]:
features.shape

(9073900, 8)

In [66]:
df['sender_out_deg'] = df['nameOrig'].map(out_degree)
df['receiver_in_deg'] = df['nameDest'].map(in_degree) 
df['sender_cluster_size'] = df['nameOrig'].map(cluster_size)
df['receiver_cluster_size'] = df['nameDest'].map(cluster_size) 
df['sender_scc_size'] = df['nameOrig'].map(scc_size)
df['receiver_scc_size'] = df['nameDest'].map(scc_size) 
df['sender_cycle'] = df['nameOrig'].map(cycle_flag)
df['receiver_cycle'] = df['nameDest'].map(cycle_flag) 
df['sender_ratio'] = df['nameOrig'].map(
    features.set_index('account')['sent_received_ratio']
) 
df['receiver_ratio'] = df['nameDest'].map(
    features.set_index('account')['sent_received_ratio']
) 

In [67]:
df.head()

,nameOrig,nameDest,amount,isFraud,sender_out_deg,receiver_in_deg,sender_cluster_size,receiver_cluster_size,sender_scc_size,receiver_scc_size,sender_cycle,receiver_cycle,sender_ratio,receiver_ratio
0,C1231006815,M1979787155,9839.64,0,1,1,2,2,1,1,0,0,9839.64,0.0
1,C1666544295,M2044282225,1864.28,0,1,1,2,2,1,1,0,0,1864.28,0.0
2,C1305486145,C553264065,181.00,1,1,44,49,49,1,1,0,0,181.00,0.0
3,C840083671,C38997010,181.00,1,1,41,42,42,1,1,0,0,181.00,0.0
4,C2048537720,M1230701703,11668.14,0,1,1,2,2,1,1,0,0,11668.14,0.0


In [ ]:
df[df['sender_cycle'] == 1]

,nameOrig,nameDest,amount,isFraud,sender_out_deg,receiver_in_deg,sender_cluster_size,receiver_cluster_size,sender_scc_size,receiver_scc_size,sender_cycle,receiver_cycle,sender_ratio,receiver_ratio


In [69]:
df.to_csv("final_map feature transactions.csv", index=False)

In [70]:
features.to_csv("features.csv", index=False)

In [71]:
df.shape

(6362620, 14)

In [72]:
features.shape

(9073900, 8)